In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
customers_clean = pd.read_parquet(
    "../data/processed/customers_clean.parquet"
)

inventory_clean = pd.read_parquet(
    "../data/processed/inventory_clean.parquet"
)

promotions_clean = pd.read_parquet(
    "../data/processed/promotions_clean.parquet"
)

sales_clean = pd.read_parquet(
    "../data/processed/sales_clean.parquet"
)

products_clean = pd.read_parquet(
    "../data/processed/products_clean.parquet"
)

stores_clean = pd.read_parquet(
    "../data/processed/stores_clean.parquet"
)

flags_clean = pd.read_parquet(
    "../data/processed/flags_clean.parquet"
)

In [5]:
sales_master = sales_clean.copy()

products_master = products_clean.copy()

customers_master = customers_clean.copy()

stores_master = stores_clean.copy()

promotions_master = promotions_clean.copy()

inventory_master = inventory_clean.copy()

In [7]:
master_df = sales_master.merge(

    products_master,

    on="sku_id",

    how="left"

)

master_df.shape

(9959019, 17)

In [9]:
print(master_df["customer_id"].dtype)

print(customers_master["cust_id"].dtype)

category
object


In [11]:
customers_master["cust_id"] = customers_master["cust_id"].astype("category")

In [13]:
master_df = master_df.merge(

    customers_master,

    left_on="customer_id",

    right_on="cust_id",

    how="left"

)

In [15]:
master_df = master_df.merge(

    stores_master,

    on="store_id",

    how="left"
)

In [17]:
master_df = master_df.merge(

    promotions_master,

    on="promo_id",

    how="left",

    suffixes=("", "_promo")
)

In [19]:
master_df = master_df.merge(

    promotions_master,

    on="promo_id",

    how="left",

    suffixes=("", "_promo")
)

In [21]:
master_df.shape

(9959019, 42)

In [23]:
master_df["year"] = master_df["date"].dt.year

master_df["month"] = master_df["date"].dt.month

master_df["quarter"] = master_df["date"].dt.quarter

master_df["day"] = master_df["date"].dt.day

master_df["weekday"] = master_df["date"].dt.day_name()

master_df["week"] = master_df["date"].dt.isocalendar().week

In [25]:
master_df["profit"] = (

    master_df["total_value"]

    -

    master_df["quantity"]

    * master_df["cost_price"]

)

In [27]:
master_df.columns[master_df.columns.duplicated()]

Index(['discount_pct_promo'], dtype='object')

In [29]:
master_df = master_df.loc[:, ~master_df.columns.duplicated()]

In [31]:
master_df.columns.duplicated().sum()

np.int64(0)

In [33]:
master_df.to_parquet(

    "../data/processed/master_df.parquet",

    index=False
)

In [35]:
master_df.shape

(9959019, 48)

In [37]:
master_df.columns

Index(['date', 'receipt_id', 'store_id', 'sku_id', 'customer_id', 'quantity',
       'unit_price_x', 'total_value', 'channel', 'discount_pct', 'promo_id',
       'sku_name', 'category', 'subcategory', 'unit_price_y', 'cost_price',
       'brand', 'cust_id', 'age', 'gender', 'city_x', 'loyalty_segment',
       'preferred_channel', 'registration_date', 'store_name', 'city_y',
       'store_type', 'opening_date', 'promo_name', 'start_date', 'end_date',
       'discount_pct_promo', 'promo_type', 'target_type', 'target_value',
       'promo_name_promo', 'start_date_promo', 'end_date_promo',
       'promo_type_promo', 'target_type_promo', 'target_value_promo', 'year',
       'month', 'quarter', 'day', 'weekday', 'week', 'profit'],
      dtype='object')

In [39]:
master_df.head()

,date,receipt_id,store_id,sku_id,customer_id,quantity,unit_price_x,total_value,channel,discount_pct,...,promo_type_promo,target_type_promo,target_value_promo,year,month,quarter,day,weekday,week,profit
0,2025-04-02,RCPT00000001,ST16,SKU02498,CUST01410,1,2379.110107,2379.11,In-Store,0.0,...,NaN,NaN,NaN,2025,4,2,2,Wednesday,14,751.20
1,2025-04-02,RCPT00000001,ST16,SKU04596,CUST01410,3,335.549988,1006.65,In-Store,0.0,...,NaN,NaN,NaN,2025,4,2,2,Wednesday,14,278.28
2,2022-04-24,RCPT00000002,ST15,SKU00078,CUST00134,1,820.530029,820.53,Online,0.0,...,NaN,NaN,NaN,2022,4,2,24,Sunday,16,167.07
3,2022-04-24,RCPT00000002,ST15,SKU00554,CUST00134,2,88.320000,176.64,Online,0.0,...,NaN,NaN,NaN,2022,4,2,24,Sunday,16,77.32
4,2024-09-22,RCPT00000003,ST20,SKU03727,CUST08826,1,1660.930054,1660.93,Online,0.0,...,NaN,NaN,NaN,2024,9,3,22,Sunday,38,369.75


In [41]:
master_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9959019 entries, 0 to 9959018
Data columns (total 48 columns):
 #   Column              Dtype         
---  ------              -----         
 0   date                datetime64[ns]
 1   receipt_id          object        
 2   store_id            object        
 3   sku_id              object        
 4   customer_id         category      
 5   quantity            int8          
 6   unit_price_x        float32       
 7   total_value         float64       
 8   channel             category      
 9   discount_pct        float32       
 10  promo_id            object        
 11  sku_name            object        
 12  category            object        
 13  subcategory         object        
 14  unit_price_y        float64       
 15  cost_price          float64       
 16  brand               object        
 17  cust_id             category      
 18  age                 int64         
 19  gender              object        
 20  ci

In [11]:
master_df = pd.read_parquet(
    "../data/processed/master_df.parquet"
)

In [13]:
master_df.drop(
    columns=[
        "cust_id",
        "unit_price_y",
        "city_y"
    ],
    inplace=True
)

In [15]:
master_df.rename(
    columns={
        "unit_price_x": "unit_price",
        "city_x": "city"
    },
    inplace=True
)

In [17]:
master_df.to_parquet(
    "../data/processed/master_df.parquet",
    compression="snappy",
    index=False
)

In [19]:
master_df.shape

(9959019, 45)

In [21]:
master_df.columns.duplicated().sum()

np.int64(0)

In [23]:
master_df.isnull().sum().sort_values(ascending=False).head(10)

promo_id              7871384
start_date_promo      7871384
end_date_promo        7871384
promo_type_promo      7871384
target_type_promo     7871384
target_value_promo    7871384
target_type           7871384
target_value          7871384
promo_name_promo      7871384
discount_pct_promo    7871384
dtype: int64

In [25]:
master_df.memory_usage(deep=True).sum() / 1024**2

np.float64(13131.197794914246)